### Create the Schema and Volume

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_project;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.ecommerce_project.ecommerce_files;

##Define the Source Path

In [0]:
source_path="/Volumes/workspace/ecommerce_project/ecommerce_files/thirupukudai_ecommerce_orders_bronze.csv"

##Read the Source Data

In [0]:
source_df=spark.read.csv(
    source_path,
    header=True,
    inferSchema=True
)
source_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+
|ORD100218|  CUST10020|    Zara Sundaram|zara.sundaram@yah...|    8249926919|  PROD1064|      

In [0]:
source_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_phone: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- order_amount: double (nullable = true)
 |-- final_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- delivery_date: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_state: string (nullable = true)
 |-- pincode: string (nullable = true)



## Define the Ingestion Information

In [0]:
batch_id="BATCH001"
pipeline_id=1
source_system="ecommerce_app"
source_file="orders_day1.csv"

## Add Bronze Metadata

In [0]:
from pyspark.sql.functions import current_timestamp,lit

bronze_df = (
    source_df
    .withColumn("ingestion_timestamp",current_timestamp())
    .withColumn("source_file",lit(source_file))
    .withColumn("batch_id",lit(batch_id))
    .withColumn("source_system",lit(source_system))
    .withColumn("pipeline_id",lit(pipeline_id))
)

In [0]:
bronze_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--

## Define the Bronze Table

In [0]:
bronze_table="workspace.ecommerce_project.bronze_orders"

### Write the Bronze Table

In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

### Read the Bronze Table using python

In [0]:
bronze_df=spark.read.table(bronze_table)
bronze_df.show()

+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+-------------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+
| order_id|customer_id|    customer_name|      customer_email|customer_phone|product_id|       product_name|   category|quantity|unit_price|discount_percent|order_amount|final_amount|  payment_method|payment_status|order_status|order_date| ship_date|delivery_date|shipping_city|shipping_state|pincode| ingestion_timestamp|    source_file|batch_id|source_system|pipeline_id|
+---------+-----------+-----------------+--------------------+--------------+----------+-------------------+-----------+--------+----------+----------------+------------+------------+----------------+--------------+------------+----------+----------+--

### Read the Bronze Table using sql

In [0]:
%sql
select * from workspace.ecommerce_project.bronze_orders;

order_id,customer_id,customer_name,customer_email,customer_phone,product_id,product_name,category,quantity,unit_price,discount_percent,order_amount,final_amount,payment_method,payment_status,order_status,order_date,ship_date,delivery_date,shipping_city,shipping_state,pincode,ingestion_timestamp,source_file,batch_id,source_system,pipeline_id
ORD100218,CUST10020,Zara Sundaram,zara.sundaram@yahoo.com,8249926919,PROD1064,Cricket Bat,Sports,8,5037.33,0,40298.64,40298.64,Net Banking,Paid,Shipped,2025-10-18,2025-10-20,null,Bengaluru,Karnataka,560073,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100164,CUST10042,Mahesh Shetty,mahesh.shetty@rediffmail.com,7678998028,PROD1073,Mouse,Electronics,4,2091.76,80,8367.04,1673.41,Net Banking,Refunded,Cancelled,2025-12-13,null,null,Lucknow,Uttar Pradesh,226094,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100064,CUST10136,Chitra Reddy,chitra.reddy@hotmail.com,8113585139,PROD1066,Books,Books,4,2731.88,10,10927.52,9834.77,Debit Card,Refunded,Cancelled,2025-11-27,null,null,Hyderabad,Telangana,500032,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100596,CUST10067,Kabir Siddiqui,kabir.siddiqui@gmail.com,8812308209,PROD1036,Water Bottle,Home,9,334.78,80,3013.02,602.6,UPI,Paid,Delivered,2025-09-06,2025-09-07,2025-09-08,Pune,Maharashtra,411008,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100313,CUST10123,Karthik Murthy,karthik.murthy@rediffmail.com,8327268503,PROD1075,Cotton Shirt,Fashion,10,2379.61,50,23796.1,11898.05,Net Banking,Paid,Delivered,2025-12-12,2025-12-15,2025-12-19,Madurai,Tamil Nadu,625013,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100660,CUST10016,Radha Chandran,radha.chandran@rediffmail.com,9389854268,PROD1048,Bluetooth Speaker,Electronics,5,7784.98,10,38924.9,35032.41,UPI,Paid,Delivered,2025-01-25,2025-01-26,2025-01-30,Jaipur,Rajasthan,302000,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100071,CUST10105,Rajesh Siddiqui,rajesh.siddiqui@rediffmail.com,6698509918,PROD1100,Mixer Grinder,Home,6,8724.76,0,52348.56,52348.56,Wallet,Paid,Delivered,2025-11-28,2025-12-01,2025-12-02,Kolkata,West Bengal,700036,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100724,CUST10036,Nandini Swamy,nandini.swamy@yahoo.com,7865523129,PROD1020,Cricket Bat,Sports,6,5502.35,60,33014.1,13205.64,Cash on Delivery,Paid,Returned,2025-11-08,2025-11-12,2025-11-16,Kolkata,West Bengal,700008,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100851,CUST10128,Rahul Subramaniam,rahul.subramaniam@yahoo.com,7608482316,PROD1096,Laptop Backpack,Electronics,10,4238.02,0,42380.2,42380.2,Wallet,Paid,Shipped,2025-08-16,null,2025-08-23,Kolkata,West Bengal,70012,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1
ORD100463,CUST10087,Vikram Murthy,vikram.murthy@yahoo.com,8141078352,PROD1049,LED Monitor,Electronics,9,38853.83,30,349684.47,244779.13,Cash on Delivery,Pending,Shipped,2025-07-19,2025-07-20,null,Lucknow,Uttar Pradesh,226096,2026-08-22T12:24:35.471Z,orders_day1.csv,BATCH001,ecommerce_app,1


### Validate the Initial Load

In [0]:
print("Before Ingestion Table count:",source_df.count())

Before Ingestion Table count: 1000


In [0]:
print("After Ingestion Table count:",bronze_df.count())

After Ingestion Table count: 1000


In [0]:
bronze_df.select(
    "ingestion_timestamp",
    "source_file",
    "batch_id",
    "source_system",
    "pipeline_id"
).show(truncate=False)

+--------------------------+---------------+--------+-------------+-----------+
|ingestion_timestamp       |source_file    |batch_id|source_system|pipeline_id|
+--------------------------+---------------+--------+-------------+-----------+
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.csv|BATCH001|ecommerce_app|1          |
|2026-08-22 12:24:35.471733|orders_day1.